# PrimeNet Fig.5 on MIMIC-IV (Colab)

Train **neutropenic fever (NF)** with native PrimeNet (TimeBERT) in **ChemoTreeVsDL**.

| Script | Role |
|--------|------|
| [`colab_primenet_fig5.py`](../colab_primenet_fig5.py) | **Fig.5 phased pipeline** (recommended) |
| [`colab_primenet_train.py`](../colab_primenet_train.py) | Single-fold smoke test |
| [`docs/PRIMENET.md`](../docs/PRIMENET.md) | Integration notes |

**Runtime:** GPU required (T4 works with `--fast`).

**Colab workflow (recommended):** run one **phase per cell** so you can reconnect between long steps:
1. `pretrain-cohort` — NF SSL pretrain (~30–60 min full, ~5 min `--fast`)
2. `finetune --scenarios nf` — 4 scenarios × 5 folds
3. Optional: `pretrain-mimicall` on `mimic_all_10pct` if you have full `mimic_all` labs on Drive

**Data:** upload NF CSVs to `data/raw/`, or copy `MIMIC_IV/saved_data/` from Drive (e.g. HPC checkpoint).

## 1. Clone repo and install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "AhmedSofan10/ChemoTreeVsDL"
BRANCH = "primenet"
REPO_DIR = "ChemoTreeVsDL"

if not Path(REPO_DIR).is_dir():
    subprocess.check_call(
        ["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR]
    )
else:
    os.chdir(REPO_DIR)
    subprocess.check_call(["git", "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "checkout", BRANCH])
    subprocess.check_call(["git", "pull", "origin", BRANCH])

os.chdir(REPO_DIR)
root = Path.cwd()
sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert (root / "ts_model_training" / "primenet" / "timebert" / "models.py").is_file()
print("Ready:", root)

## 2. Upload data (skip if using Drive `saved_data`)

`data/raw/` is **empty in git** on purpose — you upload the two CSV files here.

Required files:
- `mimic_cohort_NF_30_days.csv`
- `mimic_cohort_NF_30_days_admissions_labs_14_days.csv`

**Alternative:** skip this section and set `USE_DRIVE_SAVED_DATA = True` in the next cell if you already have `MIMIC_IV/saved_data/` on Google Drive (e.g. from BionetsProj).

In [ ]:
from pathlib import Path
from google.colab import files

RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
print("Upload the two CSV files when prompted:")
print("  - mimic_cohort_NF_30_days.csv")
print("  - mimic_cohort_NF_30_days_admissions_labs_14_days.csv")
uploaded = files.upload()
for name, data in uploaded.items():
    dest = RAW / name
    dest.write_bytes(data)
    print(f"Saved {dest} ({dest.stat().st_size // 1024} KB)")

required = [
    "mimic_cohort_NF_30_days.csv",
    "mimic_cohort_NF_30_days_admissions_labs_14_days.csv",
]
missing = [f for f in required if not (RAW / f).is_file()]
if missing:
    raise FileNotFoundError(f"Still missing in data/raw/: {missing}")
print("OK — ready to prepare saved_data")

## 3. Options

In [ ]:
# --- Fig.5 pipeline (recommended) ---
USE_FIG5 = True
RUN_FAST = True          # True = smoke timings; False = full training
SKIP_PREPARE = False     # True if MIMIC_IV/saved_data/ already exists

USE_DRIVE_SAVED_DATA = False
DRIVE_SAVED_DATA = "/content/drive/MyDrive/ChemoTreeVsDL/MIMIC_IV/saved_data"
SAVE_TO_DRIVE = True
DRIVE_OUT = "/content/drive/MyDrive/ChemoTreeVsDL/MIMIC_IV/saved_data"

# mimic_all on Colab: needs full mimic_all labs on Drive; we use 10% subset only
MIMIC_ALL_COHORT = "mimic_all_10pct"
SKIP_MIMICALL = True     # set False only if you copied full mimic_all from HPC/Drive

# --- Legacy single-fold mode (USE_FIG5 = False) ---
RUN_ALL_FOLDS = False
FOLD = 0
PREFIX = "colab_primenet"

## 4. Mount Drive / load saved_data (optional)

In [ ]:
def _mount_drive():
    from google.colab import drive
    drive.mount("/content/drive")

def _copy_saved_data_from_drive():
    import shutil
    dst = Path("MIMIC_IV/saved_data")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_SAVED_DATA, dst, dirs_exist_ok=True)
    print("Copied saved_data from Drive →", dst.resolve())

def _save_to_drive():
    import shutil
    _mount_drive()
    Path(DRIVE_OUT).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree("MIMIC_IV/saved_data", DRIVE_OUT, dirs_exist_ok=True)
    print("Saved results to Drive →", DRIVE_OUT)

if USE_DRIVE_SAVED_DATA:
    _mount_drive()
    _copy_saved_data_from_drive()
    SKIP_PREPARE = True

## 5a. Fig.5 — Phase 1: NF cohort pretrain

Run this cell first. Reuses checkpoint if `fig5_pt_cohort` already exists (e.g. copied from HPC).

In [ ]:
if USE_FIG5:
    cmd = [sys.executable, "colab_primenet_fig5.py", "--phase", "pretrain-cohort"]
    if SKIP_PREPARE:
        cmd.append("--skip-prepare")
    if RUN_FAST:
        cmd.append("--fast")
    subprocess.check_call(cmd)
    if SAVE_TO_DRIVE:
        _save_to_drive()
else:
    print("Set USE_FIG5 = True to run this cell")

## 5b. Fig.5 — Phase 2: NF finetune (4 scenarios × 5 folds)

Runs `cohort_all`, `cohort_final`, `none_none`, `none_final`. Longest Colab step when `RUN_FAST = False`.

In [ ]:
if USE_FIG5:
    cmd = [
        sys.executable,
        "colab_primenet_fig5.py",
        "--phase",
        "finetune",
        "--scenarios",
        "nf",
        "--skip-prepare",
    ]
    if RUN_FAST:
        cmd.append("--fast")
    subprocess.check_call(cmd)
    if SAVE_TO_DRIVE:
        _save_to_drive()
else:
    print("Set USE_FIG5 = True to run this cell")

## 5c. Fig.5 — Optional: mimic_all 10% pretrain + finetune

Requires full `mimic_all` lab file on Drive/HPC (`MIMIC_IV/saved_data/processed_admission_features_for_ts/mimic_all/...`). Set `SKIP_MIMICALL = False`.

In [ ]:
if USE_FIG5 and not SKIP_MIMICALL:
    for phase, scenarios in [
        ("pretrain-mimicall", None),
        ("finetune", "mimicall"),
    ]:
        cmd = [
            sys.executable,
            "colab_primenet_fig5.py",
            "--phase",
            phase,
            "--mimic-all-cohort",
            MIMIC_ALL_COHORT,
            "--skip-prepare",
            "--skip-extract-mimic-all",
        ]
        if scenarios:
            cmd += ["--scenarios", scenarios]
        if RUN_FAST:
            cmd.append("--fast")
        subprocess.check_call(cmd)
    if SAVE_TO_DRIVE:
        _save_to_drive()
else:
    print("Skipped mimicall (SKIP_MIMICALL=True or USE_FIG5=False)")

## 5d. Legacy: single-fold smoke test (`colab_primenet_train.py`)

In [ ]:
if not USE_FIG5:
    cmd = [sys.executable, "colab_primenet_train.py", "--prefix", PREFIX]
    if RUN_FAST:
        cmd.append("--fast")
    if SKIP_PREPARE:
        cmd.append("--skip-prepare")
    if RUN_ALL_FOLDS:
        cmd.append("--all-folds")
    else:
        cmd.extend(["--fold", str(FOLD)])
    subprocess.check_call(cmd)
    if SAVE_TO_DRIVE:
        _save_to_drive()
else:
    print("Fig.5 mode — use cells 5a–5b instead")

## 6. Collect Fig.5 results (optional)

In [ ]:
if USE_FIG5:
    subprocess.check_call(
        [
            sys.executable,
            "colab_primenet_fig5.py",
            "--phase",
            "collect",
            "--scenarios",
            "nf",
            "--skip-prepare",
        ]
    )
else:
    subprocess.check_call(
        [
            sys.executable,
            "scripts/summarize_folds.py",
            "--prefix",
            PREFIX,
            "--model",
            "primenet",
        ]
    )